# Outlier Detection

Identifies statistical outliers in numeric columns using `IsolationForest` (sklearn). Adds `Outlier_Flag` (`yes`/`no`) and `Outlier_Score#number` (higher = more anomalous) columns. The score distribution histogram helps you choose an appropriate threshold.

In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

SUAVE_TOKEN = ''   # @param {type:"string"}
SUAVE_HOST  = ''   # @param {type:"string"}

_p = su.load_params(token=SUAVE_TOKEN, host=SUAVE_HOST)
if _p is None:
    raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb, or enter token above.')

user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
params        = ''
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'
_prefix       = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
full_notebook_url = _prefix.rstrip('/') + '/lab/tree/SuAVEDispatch.ipynb'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

## 1. Load data and select numeric columns

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

num_cols = [c for c in df.columns if '#number' in c]
if not num_cols:
    raise ValueError('No #number columns found.')

col_selector = widgets.SelectMultiple(
    options=num_cols, value=num_cols[:min(5, len(num_cols))],
    description='Columns:', rows=min(10, len(num_cols)),
    layout=widgets.Layout(width='70%', height='200px')
)
contamination = widgets.FloatSlider(
    value=0.05, min=0.01, max=0.3, step=0.01,
    description='Expected outlier fraction:',
    continuous_update=False, layout=widgets.Layout(width='60%')
)
threshold_slider = widgets.FloatSlider(
    value=0.6, min=0.0, max=1.0, step=0.05,
    description='Flag threshold (score):',
    continuous_update=False, layout=widgets.Layout(width='60%')
)
display(col_selector, contamination, threshold_slider)

## 2. Fit IsolationForest

In [ ]:
selected = list(col_selector.value)

X = df[selected].apply(pd.to_numeric, errors='coerce')
missing = X.isnull().sum().sum()
if missing > 0:
    printmd(f"**{missing} missing values imputed with column mean.**")
    X = X.fillna(X.mean())

model   = IsolationForest(contamination=contamination.value, random_state=42)
preds   = model.fit_predict(X)           # -1 = outlier, 1 = inlier
raw_scores = model.score_samples(X)      # more negative = more anomalous

# Normalize scores to 0–1 (0 = normal, 1 = most anomalous)
scores_norm = 1 - (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())

threshold = threshold_slider.value
df['Outlier_Score#number'] = scores_norm.round(4)
df['Outlier_Flag']         = (scores_norm >= threshold).map({True: 'yes', False: 'no'})

n_flagged = (df['Outlier_Flag'] == 'yes').sum()
printmd(f"**{n_flagged} rows flagged as outliers (score >= {threshold}).**")

# Score histogram
plt.figure(figsize=(8, 4))
plt.hist(scores_norm, bins=50, color='#3498db', edgecolor='white')
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold = {threshold}')
plt.title('Outlier score distribution')
plt.xlabel('Score (0=normal, 1=most anomalous)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

printmd('**Top 10 outliers:**')
display(df.sort_values('Outlier_Score#number', ascending=False)[selected + ['Outlier_Score#number', 'Outlier_Flag']].head(10))

## 3. Adjust threshold if needed and re-flag

Change the threshold slider above and re-run this cell to update flags without refitting the model.

In [ ]:
threshold = threshold_slider.value
df['Outlier_Flag'] = (df['Outlier_Score#number'] >= threshold).map({True: 'yes', False: 'no'})
n_flagged = (df['Outlier_Flag'] == 'yes').sum()
printmd(f"**{n_flagged} rows flagged at threshold {threshold}.**")

## 4. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
input_text = widgets.Text(placeholder='Enter survey name...')
output_text = widgets.Text()

def _bind(sender):
    output_text.value = input_text.value

input_text.on_submit(_bind)
printmd("**Enter survey name, press Enter, then run the next cell:**")
display(input_text, output_text)

In [ ]:
survey_name = output_text.value
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)